In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

In [2]:
data = pd.read_csv("../Merge/final_selected_data.csv")
data.head()

,room_type_id,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,...,wifi_miễn_phí,không_hoàn_tiền,miễn_phí_hủy,vào_hồ_bơi_miễn_phí,đã_kèm_bữa_sáng,hotel_id,room_room_type_name,hotel_name,hotel_address,region
0,1,1,0,0,0,0,0,0,0,0,...,1,0,1,0,0,71897952,Phòng Deluxe Có Giường Cỡ King (Deluxe King Room),Nha Nghi Nhung - Nhung Motel,"73 Đoàn Thị Điểm, Bà Rịa, Bà Rịa, Việt Nam",Bà Rịa
1,2,0,1,0,0,0,0,0,0,0,...,1,0,1,0,0,49685029,Phòng Tiêu Chuẩn (Standard Room),Nhà nghỉ Ruby Bà Rịa (Ruby Motel Bà Rịa),"KDC Baria City Gate, Long Huong Ward, Ba Ria C...",Bà Rịa
2,3,0,1,0,0,0,0,0,0,0,...,1,0,1,0,0,49685029,Phòng gia đình có ban công (Family Room with B...,Nhà nghỉ Ruby Bà Rịa (Ruby Motel Bà Rịa),"KDC Baria City Gate, Long Huong Ward, Ba Ria C...",Bà Rịa
3,4,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,65481766,Phòng Có Giường Cỡ King Với Ban Công (King Roo...,Baly Hotel Bà Rịa City (Baly Hotel Ba Ria City),"QL51, Bà Rịa, Bà Rịa, Việt Nam",Bà Rịa
4,5,1,0,1,0,0,0,0,0,0,...,1,0,0,0,0,5808626,Phòng Studio Executive (Studio Executive),Citadines Central Bình Dương (Citadines Centra...,"Số 328C, Đại lộ B nh Dương, Khu phố Hưng Lộc, ...",Bình Dương


In [3]:
data2 = data.drop(columns=[
    'hotel_id',
    'hotel_name',
    'room_room_type_name',
    'hotel_address',
    'room_type_id', 
    'views', 'region'
])
data2.head()

,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,flexibility_score,...,price_option_price,adults_number,children_number,bãi_đậu_xe,phòng_tập,wifi_miễn_phí,không_hoàn_tiền,miễn_phí_hủy,vào_hồ_bơi_miễn_phí,đã_kèm_bữa_sáng
0,1,0,0,0,0,0,0,0,0,1,...,289522.0,2,0,1,0,1,0,1,0,0
1,0,1,0,0,0,0,0,0,0,1,...,361111.0,2,0,1,0,1,0,1,0,0
2,0,1,0,0,0,0,0,0,0,1,...,601852.0,4,2,1,0,1,0,1,0,0
3,0,1,0,0,0,0,0,0,0,1,...,425926.0,2,0,1,0,0,0,1,0,0
4,1,0,1,0,0,0,0,0,0,2,...,1450000.0,2,0,0,0,1,0,0,0,0


In [4]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
data2['sqm'] = scaler.fit_transform(data2[['sqm']])
data2['price_option_price'] = scaler.fit_transform(data2[['price_option_price']])

In [5]:
print(data2['sqm'].mean())      # ≈ 0
print(data2['sqm'].var())

3.305948255447458e-17
1.0000553924555475


In [6]:
#data2.to_csv('preprocess_final_totrain_rforest.csv')
#data2.head()

In [7]:
data2['price_option_price'] = data2['price_option_price'].fillna(data2['price_option_price'].median())
data2['sqm'] = data2['sqm'].fillna(data2['sqm'].median())

In [8]:
x_data = data2.drop(columns=['price_option_price'])
y_data = data2['price_option_price']
x_data

,large_double_bed,large_bed,single_bed,sofa_bed,double_bed,small_double_bed,king_size_bed,futon_mattress,bunk_bed,flexibility_score,...,bedroom_count,adults_number,children_number,bãi_đậu_xe,phòng_tập,wifi_miễn_phí,không_hoàn_tiền,miễn_phí_hủy,vào_hồ_bơi_miễn_phí,đã_kèm_bữa_sáng
0,1,0,0,0,0,0,0,0,0,1,...,0,2,0,1,0,1,0,1,0,0
1,0,1,0,0,0,0,0,0,0,1,...,0,2,0,1,0,1,0,1,0,0
2,0,1,0,0,0,0,0,0,0,1,...,0,4,2,1,0,1,0,1,0,0
3,0,1,0,0,0,0,0,0,0,1,...,0,2,0,1,0,0,0,1,0,0
4,1,0,1,0,0,0,0,0,0,2,...,0,2,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19515,0,0,1,0,0,0,0,0,0,1,...,0,2,0,1,0,1,0,1,0,0
19516,0,0,1,0,0,0,0,0,0,1,...,0,2,0,1,0,1,0,1,0,0
19517,0,0,0,0,1,0,0,0,0,2,...,0,4,0,1,0,1,0,1,0,0
19518,1,0,0,0,0,0,0,0,0,1,...,0,2,0,1,0,1,0,1,0,0


In [9]:
y_data

0       -0.216136
1       -0.204025
2       -0.163296
3       -0.193059
4       -0.019804
           ...   
19515   -0.227807
19516   -0.219289
19517   -0.218832
19518   -0.230825
19519   -0.217107
Name: price_option_price, Length: 19520, dtype: float64

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.25, random_state=42)

In [11]:
X_train.shape

(14640, 43)

In [12]:
X_test.shape

(4880, 43)

In [13]:
hist_gboost = HistGradientBoostingRegressor(loss='squared_error', learning_rate=0.08, max_iter=1600).fit(X_train, y_train)

In [14]:
#R2 score
hist_gboost.score(X_test, y_test)

0.11158993335373768

In [15]:

y_pred = hist_gboost.predict(X_test)
mean_squared_error(y_test, y_pred)
#MSE score

0.6165894200098881

In [16]:
#MAE score
mean_absolute_error(y_test, y_pred)

0.16140353912106797

In [17]:
from sklearn.ensemble import AdaBoostRegressor
ada_gboost = AdaBoostRegressor(n_estimators = 180, learning_rate = 0.04, loss= 'linear')
ada_gboost.fit(X_train, y_train)

,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.If ``None``, then the base estimator is:class:`~sklearn.tree.DecisionTreeRegressor` initialized with`max_depth=3`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",None
,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",180
,"learning_rate learning_rate: float, default=1.0Weight applied to each regressor at each boosting iteration. A higherlearning rate increases the contribution of each regressor. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",0.04
,"loss loss: {'linear', 'square', 'exponential'}, default='linear'The loss function to use when updating the weights after eachboosting iteration.",'linear'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.In addition, it controls the bootstrap of the weights used to train the`estimator` at each boosting iteration.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None


In [18]:
y_pred_ada = ada_gboost.predict(X_test)
print(mean_squared_error(y_test, y_pred_ada)) #MSE
print(mean_absolute_error(y_test, y_pred_ada)) #MAE
print(ada_gboost.score(X_test, y_test)) #R2

1.6254141043575006
0.21631793025537624
-1.3419705332551182


In [19]:
from sklearn.tree import DecisionTreeRegressor
decision_tree = DecisionTreeRegressor(criterion = 'squared_error', splitter='random')
decision_tree.fit(X_train, y_train)

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'random'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"m

In [20]:
y_pred_decisiontree = decision_tree.predict(X_test)
print(decision_tree.score(X_test, y_test)) #R2 score 
print(mean_absolute_error(y_test,y_pred_decisiontree)) #MAR
print(mean_squared_error(y_test, y_pred_decisiontree))#MSE

0.007857075047858708
0.14252862952304998
0.6885838574212519


In [21]:
from sklearn.linear_model import LinearRegression
linear_regression = LinearRegression(n_jobs = -1).fit(X_train, y_train)
y_pred_linearregression = linear_regression.predict(X_test)
print(linear_regression.score(X_test, y_test)) #R2 score 
print(mean_absolute_error(y_test,y_pred_linearregression)) #MAR
print(mean_squared_error(y_test, y_pred_linearregression))#MSE

0.08731030448514177
0.2071616083987533
0.6334403797684337
